# Fast DistilBERT Fine-Tuning for Hybrid Router Agent Complexity Classifier

This self-contained notebook fine-tunes **`distilbert-base-uncased`** to classify user prompts as either **`simple` (0, Local SLM)** or **`complex` (1, Cloud LLM)** using your dataset `data/training_final.csv` (1,200 examples).

### **Colab GPU Setup Reminder:**
> Before running, make sure GPU acceleration is enabled:
> 1. Click **Runtime** in the top menu -> **Change runtime type**.
> 2. Under **Hardware accelerator**, select **T4 GPU**.
> 3. Click **Save**.
> 
> On a free Colab T4 GPU, 3 epochs will take approximately **40–60 seconds** to complete.

## 1. Install Required Dependencies
Install the latest Hugging Face transformers, datasets, accelerate, and evaluation tools.

In [ ]:
!pip install -q transformers datasets torch scikit-learn accelerate

## 2. Load Dataset (`data/training_final.csv`)
Choose **Option A** (direct browser upload, easiest) or **Option B** (Google Drive mount).

In [ ]:
import os
import pandas as pd

# -------------------------------------------------------------------------
# OPTION A: Direct File Upload (RECOMMENDED)
# Click the 'Choose Files' button that appears and upload data/training_final.csv
# -------------------------------------------------------------------------
from google.colab import files
print("Please upload 'training_final.csv' from your local project's data/ folder:")
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]

# -------------------------------------------------------------------------
# OPTION B: Google Drive Mount (Alternative if you uploaded to Drive)
# -------------------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# csv_filename = '/content/drive/MyDrive/router-agent/data/training_final.csv'

# Inspect loaded dataset
df = pd.read_csv(csv_filename)
print("=" * 60)
print(f"Loaded {len(df)} rows from {csv_filename}")
print(f"Columns: {list(df.columns)}")
print("\nClass Distribution:")
print(df['label'].value_counts())
print("=" * 60)
df.head()

## 3. Preprocessing, Label Mapping & Stratified Split
- Maps `simple -> 0` (Local) and `complex -> 1` (Cloud).
- Performs a stratified 80% train / 20% validation split (random seed 42).
- Tokenizes prompts using `distilbert-base-uncased` with padding and truncation to `max_length=128`.

In [ ]:
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

# Verify GPU device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using PyTorch device: {device.upper()}")
if device == "cpu":
    print("⚠️ Warning: GPU is not active. Go to Runtime -> Change runtime type -> T4 GPU for faster training.")

# 1. Encode labels: simple=0, complex=1
label_map = {"simple": 0, "complex": 1}
df["label_id"] = df["label"].map(label_map)

# 2. Stratified 80/20 train/validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label_id"],
    random_state=42
)
print(f"Train set:      {len(train_df)} rows ({train_df['label_id'].value_counts().to_dict()})")
print(f"Validation set: {len(val_df)} rows ({val_df['label_id'].value_counts().to_dict()})")

# 3. Tokenize with distilbert-base-uncased
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["prompt"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Convert to Hugging Face Dataset format
train_ds = Dataset.from_pandas(train_df[["prompt", "label_id"]].rename(columns={"label_id": "label"}))
val_ds = Dataset.from_pandas(val_df[["prompt", "label_id"]].rename(columns={"label_id": "label"}))

train_tokenized = train_ds.map(tokenize_fn, batched=True)
val_tokenized = val_ds.map(tokenize_fn, batched=True)
print("\nTokenization complete! Sample tokens shape:", train_tokenized[0]['input_ids'][:10])

## 4. Model Setup & Training with Overfitting Detection
- 3 Epochs, batch size 16, learning rate 2e-5, AdamW optimizer with weight decay.
- Evaluates and prints training loss and validation accuracy after every epoch.
- Custom `OverfittingMonitorCallback` actively checks if validation accuracy drops or validation loss climbs while training continues.

In [ ]:
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Initialize base model with binary classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "simple", 1: "complex"},
    label2id={"simple": 0, "complex": 1}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

# Overfitting monitor callback
class OverfittingMonitorCallback(TrainerCallback):
    def __init__(self):
        self.history = []

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            epoch = state.epoch
            val_acc = metrics.get("eval_accuracy", 0.0)
            val_loss = metrics.get("eval_loss", 0.0)
            self.history.append({"epoch": epoch, "acc": val_acc, "loss": val_loss})
            
            print("\n" + "-" * 70)
            print(f"🔔 [EPOCH {epoch:.1f} EVALUATION SUMMARY]")
            print(f"   Validation Loss:     {val_loss:.4f}")
            print(f"   Validation Accuracy: {val_acc * 100:.2f}%")
            
            if len(self.history) >= 2:
                prev = self.history[-2]
                if val_acc < prev["acc"] or val_loss > prev["loss"]:
                    print(f"   ⚠️ [OVERFITTING ALERT]: Validation performance degraded vs Epoch {prev['epoch']:.1f}!")
                    print(f"      (Val Loss: {prev['loss']:.4f} -> {val_loss:.4f} | Val Acc: {prev['acc']*100:.2f}% -> {val_acc*100:.2f}%)")
                else:
                    print(f"   ✅ [HEALTHY LEARNING]: Generalization improved vs Epoch {prev['epoch']:.1f}.")
            print("-" * 70 + "\n")

training_args = TrainingArguments(
    output_dir="./distilbert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=20,
    seed=42,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[OverfittingMonitorCallback()]
)

print("Starting DistilBERT fine-tuning on GPU...")
trainer.train()

## 5. Validation Evaluation & Confusion Matrix
Computes full classification metrics (Accuracy, Precision, Recall, F1) and prints an ASCII confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Run inference on the held-out validation set
eval_results = trainer.predict(val_tokenized)
y_preds = np.argmax(eval_results.predictions, axis=-1)
y_true = eval_results.label_ids

print("=" * 65)
print("            FINAL VALIDATION SET PERFORMANCE")
print("=" * 65)
print(classification_report(
    y_true,
    y_preds,
    target_names=["Simple (0, Local)", "Complex (1, Cloud)"],
    digits=4
))

cm = confusion_matrix(y_true, y_preds)
print("\n" + "-" * 65)
print("CONFUSION MATRIX:")
print(f"                        Predicted Simple  |  Predicted Complex")
print(f"Actual Simple (Local):  {cm[0][0]:>16}  |  {cm[0][1]:>17}")
print(f"Actual Complex (Cloud): {cm[1][0]:>16}  |  {cm[1][1]:>17}")
print("-" * 65)

## 6. Save Model & Tokenizer, and Package as ZIP Archive

In [ ]:
import shutil

export_dir = "./saved_models/distilbert_classifier"
os.makedirs(export_dir, exist_ok=True)

print(f"Saving best model weights and tokenizer files to: {export_dir}")
trainer.model.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)

# Create single zip archive: saved_models/distilbert_classifier.zip
zip_base = "./saved_models/distilbert_classifier"
zip_filepath = shutil.make_archive(zip_base, "zip", export_dir)

file_size_mb = os.path.getsize(zip_filepath) / (1024 * 1024)
print("=" * 65)
print(f"✅ Successfully created archive: {zip_filepath}")
print(f"   Archive Size: {file_size_mb:.2f} MB")
print("=" * 65)

## 7. Download Model ZIP to Local Computer
Run the cell below to trigger the automatic browser download.

### **Where to place the downloaded files in your project:**
1. Extract `distilbert_classifier.zip` into your project at:
   ```
   backend/saved_models/distilbert_classifier/
   ```
2. It should contain:
   - `config.json`
   - `model.safetensors` (or `pytorch_model.bin`)
   - `tokenizer_config.json`
   - `vocab.txt`
   - `special_tokens_map.json`
3. Set `CLASSIFIER_MODE=distilbert` in your `.env` file to activate it!

In [ ]:
from google.colab import files
print("Triggering download of 'distilbert_classifier.zip'...")
files.download('saved_models/distilbert_classifier.zip')